# PixCell Stage 3 Paired Ablation Cache on Colab A100

This notebook is for growing the paired ablation cache to a large target such as `5000` tiles on a single Colab A100.

## What is the bottleneck?

The main bottleneck is GPU inference, not JSON or PNG writing.

- Each tile renders `15` ablation conditions: `4` singles, `6` pairs, `4` triples, and `1` all-groups image.
- With `--num-steps 20`, that is `15 x 20 = 300` denoising steps per tile.
- For `5000` tiles, that becomes `75,000` rendered images total.
- On a single GPU, `--jobs 2` means two Python worker processes each loading a full checkpoint and competing for the same GPU.

## Recommendation

Start with `--jobs 1` on Colab A100. It is usually the safer and often faster setting on a single GPU because it avoids VRAM duplication and worker contention.

This notebook grows the cache in chunks using `--target-total-tiles`, so it is easy to resume after a Colab disconnect.

## Runtime Notes

- Runtime type: `GPU`
- Preferred accelerator: `A100`
- This notebook assumes you have access to the PixCell repo plus the dataset and checkpoints.
- The easiest pattern is to sync inputs from S3 and sync outputs back after each chunk.

In [ ]:
import os
import subprocess
from pathlib import Path

from IPython.display import clear_output

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    userdata = None
    IN_COLAB = False

print(f"IN_COLAB={IN_COLAB}")

## Clone Repo and Install Dependencies

This follows the repo's setup guidance, but uses plain `pip` for Colab instead of a separate conda environment.

In [ ]:
%cd /content
!rm -rf /content/PixCell
!git clone https://github.com/pohaoc2/PixCell.git
%cd /content/PixCell
!git checkout main
!python -m pip install --upgrade pip setuptools==75.2.0 wheel
!python -m pip install mmcv==1.7.0 --no-build-isolation
!python -m pip install -r requirements.txt
clear_output()
%cd /content/PixCell
!nvidia-smi

## Optional: Configure AWS Credentials from Colab Secrets

If your data, checkpoints, or outputs live in S3, store these in Colab Secrets first:

- `AWS_ACCESS_KEY_ID`
- `AWS_SECRET_ACCESS_KEY`
- `AWS_DEFAULT_REGION`

If you do not use S3, you can skip this cell.

In [ ]:
if IN_COLAB and userdata is not None:
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_DEFAULT_REGION"]:
        try:
            value = userdata.get(key)
        except Exception:
            value = None
        if value:
            os.environ[key] = value

!python -m pip install awscli

## Configure Paths

Update the S3 paths if needed. The local paths match the command you have been using.

In [ ]:
REPO = Path('/content/PixCell')
DATA_ROOT = REPO / 'data' / 'orion-crc33'
CHECKPOINT_DIR = REPO / 'checkpoints' / 'pixcell_controlnet_exp' / 'npy_inputs'
OUTPUT_DIR = REPO / 'inference_output' / 'paired_ablation' / 'ablation_results'

# Optional S3 URIs. Leave input URIs as None if you are not using S3 for inputs.
S3_DATA_ROOT = None
S3_CHECKPOINT_ROOT = None
S3_OUTPUT_ROOT = None
S3_ZIP_DEST = 's3://bagherilab-working/pohao/share_space/ablation_generation/paired/'

ZIP_OUTPUT = REPO / 'inference_output' / 'paired_ablation_5000_tiles.zip'

TARGET_TOTAL_TILES = 5000
CHUNK_TARGETS = [500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000]

GUIDANCE_SCALE = 2.5
NUM_STEPS = 20
SEED = 42
TILE_SAMPLE_SEED = 42
DEVICE = 'cuda'
JOBS = 1

print(DATA_ROOT)
print(CHECKPOINT_DIR)
print(OUTPUT_DIR)

## Optional: Pull Data and Checkpoints from S3

- `S3_DATA_ROOT` can be either an S3 prefix or a single `.zip` object.
- If it is a zip file such as `s3://.../orion-crc33.zip`, the notebook downloads it and extracts it under `data/`.
- `S3_CHECKPOINT_ROOT` should remain an S3 prefix.

In [ ]:
def run(cmd, cwd=REPO):
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=str(cwd), check=True)

def pull_s3_path(source, dest):
    source = str(source)
    dest = Path(dest)
    if source.lower().endswith('.zip'):
        dest.parent.mkdir(parents=True, exist_ok=True)
        local_zip = dest.parent / Path(source).name
        run(['aws', 's3', 'cp', source, str(local_zip)])
        run(['unzip', '-q', '-o', str(local_zip), '-d', str(dest.parent)])
    else:
        dest.parent.mkdir(parents=True, exist_ok=True)
        run(['aws', 's3', 'sync', source, str(dest)])

if S3_DATA_ROOT:
    pull_s3_path(S3_DATA_ROOT, DATA_ROOT)

if S3_CHECKPOINT_ROOT:
    pull_s3_path(S3_CHECKPOINT_ROOT, CHECKPOINT_DIR)

DATA_ROOT.exists(), CHECKPOINT_DIR.exists()

## Sanity Check the Runtime

This gives you one place to confirm the Colab GPU and PyTorch CUDA state before starting a long run.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device', torch.cuda.get_device_name(0))

## Optional: Small Benchmark

Run a short sample first. Keep `JOBS=1` unless you have already proven `JOBS=2` is faster on your exact Colab runtime.

In [ ]:
SMOKE_TEST_TARGET = 5
run([
    'python', 'tools/stage3/generate_ablation_subset_cache.py',
    '--data-root', str(DATA_ROOT),
    '--checkpoint-dir', str(CHECKPOINT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--target-total-tiles', str(SMOKE_TEST_TARGET),
    '--seed', str(SEED),
    '--tile-sample-seed', str(TILE_SAMPLE_SEED),
    '--device', DEVICE,
    '--guidance-scale', str(GUIDANCE_SCALE),
    '--num-steps', str(NUM_STEPS),
    '--jobs', str(JOBS),
])

## Main Run: Grow the Paired Cache to 5000 Tiles

This uses incremental `--target-total-tiles` values so you can resume after interruptions. If the cache already has enough tiles, the script will skip that chunk automatically.

If you want a single uninterrupted run instead, use just `[5000]` for `CHUNK_TARGETS`.

In [ ]:
for target in CHUNK_TARGETS:
    if target > TARGET_TOTAL_TILES:
        continue
    print(f'\n=== Growing paired cache to {target} tiles ===')
    run([
        'python', 'tools/stage3/generate_ablation_subset_cache.py',
        '--data-root', str(DATA_ROOT),
        '--checkpoint-dir', str(CHECKPOINT_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--target-total-tiles', str(target),
        '--seed', str(SEED),
        '--tile-sample-seed', str(TILE_SAMPLE_SEED),
        '--device', DEVICE,
        '--guidance-scale', str(GUIDANCE_SCALE),
        '--num-steps', str(NUM_STEPS),
        '--jobs', str(JOBS),
    ])
    if S3_OUTPUT_ROOT:
        run(['aws', 's3', 'sync', str(OUTPUT_DIR), S3_OUTPUT_ROOT])

## Check Progress

This reports how many per-tile cache directories are present.

In [ ]:
tile_dirs = sorted([p for p in OUTPUT_DIR.iterdir() if p.is_dir() and (p / 'manifest.json').exists()]) if OUTPUT_DIR.exists() else []
print('cached tiles:', len(tile_dirs))
if tile_dirs:
    print('first tile:', tile_dirs[0].name)
    print('last tile:', tile_dirs[-1].name)

## Zip the Paired Output Folder

This creates a single zip artifact for upload. It uses `ZIP_STORED` so the process is mostly packaging I/O rather than spending extra CPU time on compression.

In [ ]:
import zipfile

source_root = OUTPUT_DIR.parent
source_name = OUTPUT_DIR.name
ZIP_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_OUTPUT, 'w', compression=zipfile.ZIP_STORED) as zf:
    for path in sorted(OUTPUT_DIR.rglob('*')):
        if path.is_file():
            zf.write(path, arcname=str(Path(source_name) / path.relative_to(OUTPUT_DIR)))

print(f'Wrote zip: {ZIP_OUTPUT}')
print(f'Size (GB): {ZIP_OUTPUT.stat().st_size / (1024 ** 3):.2f}')

## Upload the Zip to S3

This uploads the zipped paired ablation folder to:

`s3://bagherilab-working/pohao/share_space/ablation_generation/paired/`

In [ ]:
run(['aws', 's3', 'cp', str(ZIP_OUTPUT), S3_ZIP_DEST])

## Next Step After Paired Reaches 5000

Once the paired cache is complete, move back to your unpaired flow:

```bash
python tools/stage3/prepare_unpaired_ablation_dataset.py \
  --paired-cache-root inference_output/paired_ablation/ablation_results \
  --data-root data/orion-crc33 \
  --metadata-only \
  --mapping-output inference_output/unpaired_ablation/metadata/unpaired_mapping.json \
  --seed 42
```

Then regenerate unpaired from scratch using that new mapping JSON.